In [ ]:
# =============================================================================
# SKY INNOVATORS PRODUCTION PLATFORM 
# Pure SegFormer-B4 Backbone | 8-Class | 3 Balanced Core Datasets
# =============================================================================

import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

import gc, sys, time, random
import numpy as np
import cv2
from PIL import Image
from glob import glob
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
import albumentations as A
from albumentations.pytorch import ToTensorV2
from transformers import SegformerForSemanticSegmentation

# ── ENVIRONMENT INITIALIZATION ────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True

def seed_everything(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
seed_everything(42)

print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# ── OPTIMIZED LANDSCAPE INTERFACE PARAMETERS ──────────────────────────────────
NUM_CLASSES   = 8  # FIXED: Dropped Fire class
IMG_SIZE      = 640
BATCH_SIZE    = 2
ACCUM_STEPS   = 4
EPOCHS        = 40
BASE_LR       = 6e-5
PATIENCE      = 12
MIN_DELTA     = 1e-4
WEIGHT_DECAY  = 0.01
IGNORE_INDEX  = -100

# Local model asset assignment targets Kaggle system directory natively
SEGFORMER_LOCAL    = "/kaggle/input/models/nvidia/segformer/pytorch/b4-finetuned-ade-512-512/1"
SEGFORMER_MODEL_ID = "nvidia/segformer-b4-finetuned-ade-512-512"
SAVE_PATH          = "/kaggle/working/sky_innovators_v10_landscape.pth"

CLASS_NAMES = [
    "Background", "HealthyTree", "DeadTree", "LowVegetation",
    "BareSoil", "Water", "Road", "Building"
]

CLASS_COLORS = np.array([
    [  0,   0,   0], [ 34, 139,  34], [139,  90,  43], [154, 205,  50],
    [205, 133,  63], [ 30, 144, 255], [169, 169, 169], [220,  80,  80]
], dtype=np.uint8)

# Class Weights rebalanced to focus heavy attention on forestry classes without Fire skew
CLASS_WEIGHTS = torch.tensor(
    [1.0, 1.5, 1.5, 1.0, 1.0, 1.0, 1.0, 1.0],
    dtype=torch.float32).to(device)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ── PATH MAPS ─────────────────────────────────────────────────────────────────
SD_IMG  = "/kaggle/input/datasets/awsaf49/semantic-drone-dataset/semantic_drone_dataset/training_set/images"
SD_MASK = "/kaggle/input/datasets/awsaf49/semantic-drone-dataset/semantic_drone_dataset/training_set/gt/semantic/label_images"
UAVID   = "/kaggle/input/datasets/dasmehdixtr/uavid-v1"
DEAD_I  = "/kaggle/input/datasets/meteahishali/aerial-imagery-for-standing-dead-tree-segmentation/USA_segmentation/RGB_images"
DEAD_M  = "/kaggle/input/datasets/meteahishali/aerial-imagery-for-standing-dead-tree-segmentation/USA_segmentation/masks"

# ── HASH CONVERSION DICTIONARIES ──────────────────────────────────────────────
DRONE_RGB_TO_CLASS = {
    (128,64,128):1,(130,76,0):2,(0,102,0):3,(112,103,87):4,(28,42,168):5,
    (48,41,30):6,(0,50,89):7,(107,142,35):8,(70,70,70):9,(102,102,156):10,
    (254,228,12):11,(254,148,12):12,(190,153,153):13,(153,153,153):14,
    (255,22,96):15,(102,51,0):16,(9,143,150):17,(119,11,32):18,
    (51,51,0):19,(190,250,190):20,(112,150,146):21,(2,135,115):22}

DRONE_REMAP = {
    0:0,1:6,2:4,3:3,4:6,5:5,6:4,7:5,8:3,9:7,10:7,11:7,12:7,
    13:0,14:0,15:0,16:0,17:0,18:0,19:1,20:2,21:0,22:0}

_DH = np.zeros(256**3, dtype=np.uint8)
for (r,g,b),o in DRONE_RGB_TO_CLASS.items(): _DH[r*65536+g*256+b] = DRONE_REMAP[o]

_UH = np.zeros(256**3, dtype=np.uint8)
for (r,g,b),c in {(0,0,0):0,(0,128,0):1,(128,128,0):3,(128,0,0):7,
                   (128,64,128):6,(64,0,128):0,(64,64,0):0,(192,0,192):0}.items():
    _UH[r*65536+g*256+b] = c

# ── TRANSFORM INTERFACES ──────────────────────────────────────────────────────
train_tf = A.Compose([
    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.3), A.RandomRotate90(p=0.4),
    A.Affine(scale=(0.85,1.15), translate_percent=0.10, rotate=(-44,44), p=0.5),
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=(-0.25,0.20), contrast_limit=(-0.20,0.20)),
        A.CLAHE(clip_limit=3.0),
    ], p=0.6),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2(),
])
val_tf = A.Compose([A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()])

# =============================================================================
# STORAGE CLASSES
# =============================================================================
class CachedDS(Dataset):
    def __init__(self, transform, name=""):
        self.transform = transform
        self.images, self.masks, self.name = [], [], name
    def add(self, img, mask):
        self.images.append(img); self.masks.append(mask)
    def __len__(self): return len(self.images)
    def __getitem__(self, idx):
        img, mask = self.images[idx], self.masks[idx]
        if (mask == IGNORE_INDEX).any():
            ign   = (mask == IGNORE_INDEX).astype(np.uint8)
            valid = np.where(ign, 0, mask).astype(np.uint8)
            aug   = self.transform(image=img, mask=valid)
            msk_t = aug["mask"].long()
            ign_r = cv2.resize(ign, (msk_t.shape[1], msk_t.shape[0]), interpolation=cv2.INTER_NEAREST)
            msk_t[torch.from_numpy(ign_r.astype(bool))] = IGNORE_INDEX
        else:
            aug   = self.transform(image=img, mask=mask.astype(np.uint8))
            msk_t = aug["mask"].long()
        return {"pixel_values": aug["image"], "labels": msk_t}

def _ri(img): return cv2.resize(img,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_LINEAR)
def _rm(m):   return cv2.resize(m.astype(np.float32),(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_NEAREST).astype(np.int16)

# =============================================================================
# DATA LOADERS PIPELINE
# =============================================================================
def build_master_dataloaders():
    print(f"\n📦 Loading pure {NUM_CLASSES}-class landscape pipeline into RAM...")
    t0 = time.time()
    
    # 1. Semantic Drone Split
    all_sd = sorted(glob(os.path.join(SD_IMG,"*.jpg")))
    random.shuffle(all_sd)
    n_val  = max(1, int(len(all_sd)*0.20))
    sd_val, sd_train = all_sd[:n_val], all_sd[n_val:]
    
    def _load_sd(paths, tf, name):
        ds = CachedDS(tf, name)
        midx = {os.path.splitext(f)[0].lower(): os.path.join(SD_MASK,f)
                for f in os.listdir(SD_MASK) if f.lower().endswith('.png')}
        print(f"  🧠 {name} ({len(paths)})...")
        for p in tqdm(paths, leave=False, mininterval=10):
            stem = os.path.splitext(os.path.basename(p))[0].lower()
            if stem not in midx: continue
            img  = _ri(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB))
            m    = np.array(Image.open(midx[stem]).convert("RGB"))
            r,g,b = (m[:,:,c].astype(np.int32) for c in range(3))
            ds.add(img, _rm(_DH[r*65536+g*256+b].astype(np.int16)))
        return ds
        
    tr_sd = _load_sd(sd_train, train_tf, "SemanticDrone-train")
    vl_sd = _load_sd(sd_val,   val_tf,   "SemanticDrone-val")

    # 2. UAVid Split
    all_uavid = []
    for split in ["uavid_train","uavid_val"]:
        sdir = os.path.join(UAVID,split)
        if not os.path.exists(sdir): continue
        for seq in sorted(glob(os.path.join(sdir,"seq*"))):
            idir, ldir = os.path.join(seq,"Images"), os.path.join(seq,"Labels")
            if not (os.path.exists(idir) and os.path.exists(ldir)): continue
            for f in os.listdir(idir):
                if f.lower().endswith('.png'):
                    lp = os.path.join(ldir, f)
                    if os.path.exists(lp):   
                        all_uavid.append((os.path.join(idir,f), lp))
                        
    random.shuffle(all_uavid)
    n_vl = max(1, int(len(all_uavid)*0.10))
    uavid_val, uavid_train = all_uavid[:n_vl], all_uavid[n_vl:]
    
    def _load_uavid(pairs, tf, name):
        ds = CachedDS(tf, name)
        print(f"  🧠 {name} ({len(pairs)})...")
        for ip,mp in tqdm(pairs, leave=False, mininterval=10):
            img = _ri(cv2.cvtColor(cv2.imread(ip), cv2.COLOR_BGR2RGB))
            m   = np.array(Image.open(mp).convert("RGB"))
            r,g,b = (m[:,:,c].astype(np.int32) for c in range(3))
            ds.add(img, _rm(_UH[r*65536+g*256+b].astype(np.int16)))
        return ds
        
    tr_uavid = _load_uavid(uavid_train, train_tf, "UAVid-train")
    vl_uavid = _load_uavid(uavid_val,   val_tf,   "UAVid-val")

    # 3. USA Standing Dead Tree
    all_dead = [(os.path.join(DEAD_I,f), os.path.join(DEAD_M,f"mask_{f[4:]}"))
                for f in os.listdir(DEAD_I)
                if f.startswith("RGB_") and f.lower().endswith('.png')
                and os.path.exists(os.path.join(DEAD_M,f"mask_{f[4:]}"))]
    random.shuffle(all_dead)
    n_vl_d = max(1, int(len(all_dead)*0.10))
    dead_val, dead_train = all_dead[:n_vl_d], all_dead[n_vl_d:]
    
    def _load_dead(pairs, tf, name):
        ds = CachedDS(tf, name)
        print(f"  🧠 {name} ({len(pairs)})...")
        for ip,mp in tqdm(pairs, leave=False, mininterval=10):
            img  = _ri(cv2.cvtColor(cv2.imread(ip), cv2.COLOR_BGR2RGB))
            gray = np.array(Image.open(mp).convert("L"))
            mask = np.full(gray.shape, IGNORE_INDEX, dtype=np.int16)
            mask[gray>0] = 2
            ds.add(img, _rm(mask))
        return ds
        
    tr_dead = _load_dead(dead_train, train_tf, "USADead-train")
    vl_dead = _load_dead(dead_val,   val_tf,   "USADead-val")

    # Amalgamate all structural data layers cleanly (FLAME3 Streams completely removed)
    train_pool = ConcatDataset([tr_sd, tr_uavid, tr_dead])
    val_pool   = ConcatDataset([vl_sd, vl_uavid, vl_dead])
    
    print(f"\n  ✅ Cache done ({time.time()-t0:.0f}s)")
    print(f"     Train : {len(train_pool):,}  (SD + UAVid + USADead)")
    print(f"     Val   : {len(val_pool):,}  (SD + UAVid + USADead)")
    
    is_cu = torch.cuda.is_available()
    tr_loader = DataLoader(train_pool, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=is_cu)
    vl_loader = DataLoader(val_pool,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=is_cu)
    return tr_loader, vl_loader

# =============================================================================
# MODEL ARCHITECTURE HUBS
# =============================================================================
def build_segformer():
    model_source = (SEGFORMER_LOCAL if os.path.exists(SEGFORMER_LOCAL) else SEGFORMER_MODEL_ID)
    print(f"🎬 Loading SegFormer-B4 from: {model_source}")
    return SegformerForSemanticSegmentation.from_pretrained(
        model_source,
        num_labels=NUM_CLASSES,
        id2label={i:n for i,n in enumerate(CLASS_NAMES)},
        label2id={n:i for i,n in enumerate(CLASS_NAMES)},
        ignore_mismatched_sizes=True,
    ).to(device)

def save_ckpt(mdl, opt, epoch, best_miou, iou_list):
    torch.save({
        "epoch": epoch, "model_state_dict": mdl.state_dict(),
        "optimizer_state_dict": opt.state_dict(),
        "best_miou": best_miou, "per_class_iou": iou_list,
        "num_classes": NUM_CLASSES, "class_names": CLASS_NAMES,
    }, SAVE_PATH)

# =============================================================================
# PYTORCH BALANCED MULTICLASS CE + DICE INTERFACES
# =============================================================================
def loss_fn(pred, tgt):
    ce   = F.cross_entropy(pred, tgt, weight=CLASS_WEIGHTS, ignore_index=IGNORE_INDEX)
    soft = F.softmax(pred, dim=1)
    tgt_oh = F.one_hot(torch.clamp(tgt,0,NUM_CLASSES-1), NUM_CLASSES).permute(0,3,1,2).float()
    vm     = (tgt != IGNORE_INDEX).unsqueeze(1)
    inter  = torch.sum(soft*vm * tgt_oh*vm, dim=(0,2,3))
    denom  = torch.sum((soft+tgt_oh)*vm,    dim=(0,2,3))
    dice   = torch.sum((1-(2*inter+1e-6)/(denom+1e-6)) * CLASS_WEIGHTS) / CLASS_WEIGHTS.sum()
    return 0.5*ce + 0.5*dice

def _upd(tp, union, p, t):
    pf,tf = p.ravel(), t.ravel()
    v = tf != IGNORE_INDEX
    pv,tv = pf[v],tf[v]
    for c in range(NUM_CLASSES):
        tp[c]    += np.sum((pv==c)&(tv==c))
        union[c] += np.sum((pv==c)|(tv==c))

def _iou(tp,union):
    iou = np.where(union>0, tp/(union+1e-9), np.nan)
    return iou, float(np.nanmean(iou))

def print_iou(iou, ep):
    print(f"\n  Per-class IoU — Ep {ep}")
    print(f"  {'Class':<16}{'IoU':>6}  Bar")
    print(f"  {'─'*44}")
    for n,v in zip(CLASS_NAMES,iou):
        if np.isnan(v):
            print(f"    {n:<14}   n/a  (no val samples yet)")
            continue
        flg = "✅" if v>=0.60 else "⚠️ " if v<0.40 else "  "
        print(f"  {flg}{n:<14} {v:>6.4f}  {'█'*int(v*20)}")
    print()

# =============================================================================
# GRADIENT TRACKING OPTIMIZERS
# =============================================================================
def make_optimizer(mdl, freeze_mode=True):
    enc = [p for n,p in mdl.segformer.named_parameters() if p.requires_grad]
    dec = [p for n,p in mdl.decode_head.named_parameters() if p.requires_grad]
    enc_lr = BASE_LR*0.1  if freeze_mode else BASE_LR*0.5
    dec_lr = BASE_LR*2.0  if freeze_mode else BASE_LR*4.0
    return torch.optim.AdamW([
        {"params": enc, "lr": enc_lr},
        {"params": dec, "lr": dec_lr},
    ], weight_decay=WEIGHT_DECAY)

def make_sched(opt, n):
    return torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n, eta_min=BASE_LR*0.01)

try:    scaler = torch.amp.GradScaler("cuda")
except: scaler = torch.cuda.amp.GradScaler()

# =============================================================================
# LIVE RUNTIME ITERATION FUNCTIONS
# =============================================================================
def train_epoch(mdl, loader, opt, ep):
    mdl.train(); tl=0.0
    tp=np.zeros(NUM_CLASSES,np.float64); union=np.zeros(NUM_CLASSES,np.float64)
    opt.zero_grad(); is_cu=device.type=="cuda"
    for i,batch in enumerate(tqdm(loader, desc=f"  Ep{ep:02d}[train]", mininterval=15, leave=False)):
        imgs=batch["pixel_values"].to(device,non_blocking=True)
        msks=batch["labels"].to(device,non_blocking=True)
        if torch.all(msks==IGNORE_INDEX): del imgs,msks; continue
        with torch.amp.autocast(device_type="cuda",enabled=is_cu):
            out = mdl(pixel_values=imgs).logits
            log = F.interpolate(out,size=imgs.shape[2:], mode="bilinear",align_corners=False)
            loss= loss_fn(log,msks)/ACCUM_STEPS
        (scaler.scale(loss) if is_cu else loss).backward()
        if (i+1)%ACCUM_STEPS==0 or (i+1)==len(loader):
            if is_cu:
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(mdl.parameters(),1.0)
                scaler.step(opt); scaler.update()
            else:
                nn.utils.clip_grad_norm_(mdl.parameters(),1.0); opt.step()
            opt.zero_grad()
        tl+=loss.item()*ACCUM_STEPS
        _upd(tp,union, log.argmax(1).detach().cpu().numpy(), msks.detach().cpu().numpy())
        del out, log, loss, imgs, msks
    _,miou=_iou(tp,union); return tl/len(loader),miou

@torch.no_grad()
def validate(mdl, loader):
    mdl.eval(); vl=0.0
    tp=np.zeros(NUM_CLASSES,np.float64); union=np.zeros(NUM_CLASSES,np.float64)
    is_cu=device.type=="cuda"
    for batch in loader:
        imgs=batch["pixel_values"].to(device,non_blocking=True)
        msks=batch["labels"].to(device,non_blocking=True)
        with torch.amp.autocast(device_type="cuda",enabled=is_cu):
            out = mdl(pixel_values=imgs).logits
            log = F.interpolate(out,size=imgs.shape[2:], mode="bilinear",align_corners=False)
            vl+=loss_fn(log,msks).item()
        _upd(tp,union,log.argmax(1).cpu().numpy(),msks.cpu().numpy())
        del out, log, imgs, msks
    iou,miou=_iou(tp,union); return vl/len(loader),miou,iou

# =============================================================================
# MAIN LIFECYCLE LIFTOFF SWITCHBOARD
# =============================================================================
if __name__=="__main__":
    train_loader, vl_loader = build_master_dataloaders() 
    model = build_segformer()
    
    print("\n❄️  Freezing encoder blocks 0-3 for epochs 1-5...")
    for n,p in model.named_parameters():
        if "encoder.block" in n: p.requires_grad = False
        
    opt   = make_optimizer(model, freeze_mode=True)
    sched = make_sched(opt, EPOCHS)
    best  = 0.0; pctr = 0
    
    print(f"\n{'='*70}")
    print(f"  SKY INNOVATORS v10.0 CLEAN |  8 classes  |  {EPOCHS} epochs")
    print(f"  Domain Optimized Matrix: Urban Infrastructure, Rural & Forest Biomass")
    print(f"{'='*70}")
    print(f"{'Ep':>4} {'TLoss':>8} {'TmIoU':>8} {'VLoss':>8} {'VmIoU':>8} {'t':>4}  Note")
    print("─"*68)
    
    for ep in range(1,EPOCHS+1):
        t0=time.time()
        if ep==6:
            print("\n🔓 Unfreezing all encoder blocks...")
            for n,p in model.named_parameters(): p.requires_grad=True
            opt   = make_optimizer(model, freeze_mode=False)
            sched = make_sched(opt, EPOCHS-ep+1)
            
        tl,tm = train_epoch(model, train_loader, opt, ep)
        vl,vm,iou = validate(model, vl_loader)
        sched.step()
        
        note=""
        if vm>best+MIN_DELTA:
            best=vm; pctr=0
            save_ckpt(model,opt,ep,best,iou.tolist()); note="✅ saved"
        else:
            pctr+=1; note=f"⏱{pctr}/{PATIENCE}"
            
        print(f"{ep:>4} {tl:>8.4f} {tm:>8.4f} {vl:>8.4f} {vm:>8.4f} {time.time()-t0:>3.0f}s  {note}")
        if ep%5==0 or ep==EPOCHS: print_iou(iou,ep)
        if pctr>=PATIENCE:
            print(f"\n🛑 Early stop ep {ep}. Best={best:.4f}"); break
            
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        
    print(f"\n✅ Done. Best mIoU={best:.4f}  Saved: {SAVE_PATH}")
